# MNIST分類タスク
Whittington modelをもとに、LA_model model、PCRHL modelを作成した。\
Whittington model   ：FBがFFの転置\
LA_model model          ：FBがランダム固定行列\
PCRHL model         ：FF、FBの両方が学習

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.ticker as ticker
from datetime import datetime
import os
import h5py
import random
from modules.func import LINEAR, TANH, RELU, LOGSIG
from modules.model import Whittington_Model,LA_model
from modules.batch import expand_one_hot_vector,reduct_pred

#MNISTのDL
from keras.datasets import mnist
(train_image, train_label), (test_image, test_label) = mnist.load_data()
train_image = train_image / 255.0
test_image  = test_image / 255.0

random.seed(42)

In [ ]:
# トライアル数
num_trial  = 10
# 学習条件
num_step   = 1*10**5    # step数
num_test   = 500        # 正答率テスト回数(1回のテストあたりの回数)
num_loop   = 50         # loop回数
# モデルの設定
x_dim      = [784,100, 10]  #細胞数
num_batch  = 20             # batch数
test_trial = 40     #正答率テスト回数(1回の実行あたりのテスト回数)
# 学習率
lr_x       = 0.125 # xの学習率(更新率)
lr_weight  = 0.010 # weightの学習率

# 活性化関数
f = LINEAR #LINEAR, TANH, RELU, LOGSIG

# list
energy_list   = []
accuracy_list = []
cossim_list    = []

for _tri in range(num_trial):   # num_trial回繰り返す

    # モデルの選択
    model = LA_model(x_dim, num_batch, lr_x, lr_weight, f) #Whittington, LA_model

    # 一時listの初期化
    energy_list_tmp = []
    accuracy_list_tmp = []
    cossim_list_tmp = []

    for step in tqdm(range(int((num_step)/num_batch))):                 # num_step回繰り返す
        idx    = np.random.randint(0, train_image.shape[0], num_batch)  # randomなMNIST画像の選択
        inputs = train_image[idx].reshape(num_batch , -1)               # inputsにrandomなMNIST画像の代入
        target = expand_one_hot_vector(train_label[idx],num_batch,x_dim[-1])    # batch処理用にvector変換

        model.initialize_states()                   # 活動の初期化
        model.update_x(inputs, target, num_loop)    # xの更新
        model.update_weight()                       # wの更新

        cossim_list_tmp.append(model.save_cossim())         #コサイン類似度記録
        energy_list_tmp.append(model.calculate_energy())    #エネルギー記録

        # 正答率評価
        if (step + 1) % int((num_step)/num_batch/test_trial) == 0 or step == 0:
            cnt = 0
            for _test in range(int(num_test/num_batch)):
                idx    = np.random.randint(0, test_image.shape[0], num_batch)
                inputs = test_image[idx].reshape(num_batch , -1)
                model.initialize_states()
                pred_vec= model.predict(inputs)
                pred_list = reduct_pred(pred_vec, num_batch, x_dim[-1])
                cnt += np.count_nonzero(pred_list == test_label[idx])
            accuracy_list_tmp.append(cnt / num_test)

    # 結果の格納
    energy_list.append(energy_list_tmp)
    accuracy_list.append(accuracy_list_tmp)
    cossim_list.append(cossim_list_tmp)


In [ ]:
import pandas as pd

def save_ndarray_to_excel(array: np.ndarray, filename: str) -> None:
    df = pd.DataFrame(array)
    df.to_excel(filename, index=False, header='Value', engine='openpyxl')

def save_ndarray_to_csv(array: np.ndarray, filename: str) -> None:
    df = pd.DataFrame(array)
    df.to_csv(filename, index=False, header='Value')

def save_string_to_txt(string_data, filename):
    with open(filename, 'w') as f:
        f.write(string_data)

In [ ]:
#Save Files
Setting="PCLA"

name = f"LINEAR_3layer_{num_trial}trial_{int(num_step)}step_20260127"
save_ndarray_to_csv(np.array(energy_list)  , f'{Setting}_E_{name}.csv')
save_ndarray_to_csv(np.array(accuracy_list), f'{Setting}_A_{name}.csv')
save_ndarray_to_csv(np.array(cossim_list)  , f'{Setting}_Cm1_{name}.csv')

## 保存

In [ ]:
# 日時の取得
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
func = f.get_name()
filename = f"{num_trial}trial_{num_step}step_{func}_{timestamp}"

# 保存
save_dir = "./MNIST_result"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, f"result_{filename}.h5")

with h5py.File(save_path, "w") as f:
    f.create_dataset("energy", data=np.array(energy_list))
    f.create_dataset("accuracy", data=np.array(accuracy_list))
    f.create_dataset("cossim", data=np.array(cossim_list))


## 読み込み

In [ ]:
# 読み込み
results = {}

file_name = f"result_{filename}.h5"
save_dir = "./MNIST_result"
load_path = os.path.join(save_dir, file_name)

with h5py.File(save_path, "r") as f:
    energy_list = f["energy"][:]
    accuracy_list    = f["accuracy"][:]
    cossim_list = f["cossim"][:]


## 可視化

In [ ]:
accuracy_result = np.array(accuracy_list)
energy_result  = np.array(energy_list)
cossim_result   = np.array(cossim_list)

# トライアルの平均、標準偏差をとる関数
def trial_mean(data):
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)
    return mean, std

# 移動平均をとる関数
def MA(data, window_size=100):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

def plot_result(data, ylabel, title, window_size=1, ylim=None):
    mean, std = trial_mean(data) # 平均と標準偏差の計算
    mean = MA(mean, window_size) 
    std  = MA(std, window_size)
    
    x = np.arange(len(mean))
    xticks = np.linspace(0, len(mean), num=6, dtype=int)  # 6分割等間隔
    xticklabels = np.linspace(0, num_step, num=6, dtype=int)  # 表示ラベルも等間隔

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(x, mean, color='black', label='Average') # 平均のプロット
    ax.fill_between(x, mean - std, mean + std, color='black', alpha=0.2, label='±1 Std. Dev.') # 標準偏差のプロット

    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=10, integer=True))
    ax.set_xlabel("Steps", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.tick_params(axis='both', labelsize=10)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_ylim(ylim) 
    ax.set_xlim(0, len(mean))
    plt.tight_layout()
    plt.show()

# plotするデータ、設定
plot_result(accuracy_result,
            ylabel="Accuracy",
            title="Accuracy Over Time",
            window_size=1,
            ylim=(0, 1))

plot_result(energy_result,
            ylabel="Energy",
            title="Energy Over Time",
            window_size=100)

plot_result(cossim_result,
            ylabel="Cosine Similarity",
            title="Cosine Similarity Over Time",
            window_size=100,
            ylim=(-1.0, 1.0))